# BirdCLEF+ 2026 — Perch v2 Submission (TFLite)

Perch v2 TFLite + LogisticRegression による提出用ノートブック。

## Prerequisites
Training notebook の Output を Dataset として追加（以下3ファイルを含む）:
- `perch_v2.tflite` — TFLite変換済みPerchモデル
- `perch_v2_classifier.pkl` — 学習済み分類器 + LabelEncoder

## Kaggle Settings
| Setting | Value |
|---------|-------|
| Accelerator | None (CPU) |
| Internet | **OFF** (required for submission) |

In [ ]:
import os
import glob
import pickle
import warnings

import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print(f'TensorFlow: {tf.__version__}')

In [ ]:
# ============================================================
# Path Configuration
# ============================================================

# Competition data
COMP_DATA = None
for p in ['/kaggle/input/birdclef-2026',
          '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(p):
        COMP_DATA = p
        break
assert COMP_DATA, 'Competition data not found.'

# TFLite model (from training notebook output)
TFLITE_PATH = None
for f in glob.glob('/kaggle/input/**/*.tflite', recursive=True):
    TFLITE_PATH = f
    break
assert TFLITE_PATH, 'perch_v2.tflite not found. Add training notebook output as Input.'

# Classifier pkl (from training notebook output)
CLF_PATH = None
for f in glob.glob('/kaggle/input/**/*classifier*.pkl', recursive=True):
    CLF_PATH = f
    break
assert CLF_PATH, 'perch_v2_classifier.pkl not found.'

SAMPLE_RATE = 32000
DURATION = 5
WINDOW_SAMPLES = SAMPLE_RATE * DURATION
OUTPUT_DIR = '/kaggle/working'

print(f'Competition data: {COMP_DATA}')
print(f'TFLite model:     {TFLITE_PATH}')
print(f'Classifier:       {CLF_PATH}')

In [ ]:
# ============================================================
# Load TFLite Perch model
# ============================================================
interpreter = tf.lite.Interpreter(
    model_path=TFLITE_PATH, num_threads=4
)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f'Input:  {input_details[0]["shape"]}')
for od in output_details:
    print(f'Output: {od["name"]} shape={od["shape"]}')

# Find the embedding output index (1536-dim for Perch v2)
EMBED_IDX = None
for od in output_details:
    # Test with dummy input to find 1536-dim output
    interpreter.set_tensor(
        input_details[0]['index'],
        np.zeros((1, WINDOW_SAMPLES), dtype=np.float32)
    )
    interpreter.invoke()
    result = interpreter.get_tensor(od['index'])
    if result.shape[-1] == 1536:
        EMBED_IDX = od['index']
        print(f'Embedding index: {EMBED_IDX} (shape {result.shape})')
        break

# Fallback: if no 1536-dim found, try largest output
if EMBED_IDX is None:
    sizes = []
    for od in output_details:
        result = interpreter.get_tensor(od['index'])
        sizes.append((od['index'], result.shape, result.shape[-1]))
    sizes.sort(key=lambda x: x[2], reverse=True)
    EMBED_IDX = sizes[0][0]
    print(f'Fallback embed index: {EMBED_IDX} (shape {sizes[0][1]})')


def embed_fn(waveform):
    """Extract embedding from 5-second waveform via TFLite."""
    interpreter.set_tensor(
        input_details[0]['index'],
        waveform.reshape(1, -1).astype(np.float32)
    )
    interpreter.invoke()
    emb = interpreter.get_tensor(EMBED_IDX)
    if emb.ndim > 2:
        emb = emb.reshape(-1, emb.shape[-1]).mean(axis=0)
    elif emb.ndim == 2:
        emb = emb[0]
    return emb.astype(np.float32)


# Verify
test_emb = embed_fn(np.zeros(WINDOW_SAMPLES, dtype=np.float32))
print(f'Test embedding shape: {test_emb.shape}')
print('TFLite Perch loaded OK')

In [ ]:
# ============================================================
# Load classifier & label encoder
# ============================================================
with open(CLF_PATH, 'rb') as f:
    data = pickle.load(f)
clf = data['clf']
le = data['le']
n_classes = len(le.classes_)
print(f'Classifier loaded: {n_classes} classes')

# Sample submission
sample_sub = pd.read_csv(f'{COMP_DATA}/sample_submission.csv')
species_cols = [c for c in sample_sub.columns if c != 'row_id']

species_to_idx = {}
for sp in species_cols:
    try:
        species_to_idx[sp] = int(le.transform([sp])[0])
    except ValueError:
        pass
print(f'Mapped: {len(species_to_idx)} / {len(species_cols)} species')

In [ ]:
# ============================================================
# Inference on test soundscapes
# ============================================================
test_dir = f'{COMP_DATA}/test_soundscapes'
test_files = sorted(f for f in os.listdir(test_dir) if f.endswith('.ogg'))
print(f'Test files: {len(test_files)}')

rows = []
for fname in tqdm(test_files, desc='Inference'):
    path = os.path.join(test_dir, fname)
    stem = os.path.splitext(fname)[0]
    
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        full_waveform, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    
    if len(full_waveform) == 0:
        continue
    
    for start in range(0, len(full_waveform), WINDOW_SAMPLES):
        window = full_waveform[start:start + WINDOW_SAMPLES]
        if len(window) < WINDOW_SAMPLES:
            window = np.pad(window, (0, WINDOW_SAMPLES - len(window)))
        window = window.astype(np.float32)
        
        bucket_sec = (start // WINDOW_SAMPLES) * DURATION
        row_id = f'{stem}_{bucket_sec}'
        
        emb = embed_fn(window)
        raw_probs = clf.predict_proba(emb.reshape(1, -1))[0]
        
        # Map to full class set
        full_probs = np.zeros(n_classes, dtype=np.float32)
        for i, c in enumerate(clf.classes_):
            full_probs[c] = raw_probs[i]
        
        row = {'row_id': row_id}
        for sp in species_cols:
            if sp in species_to_idx:
                row[sp] = float(full_probs[species_to_idx[sp]])
            else:
                row[sp] = 0.0
        rows.append(row)

print(f'Total predictions: {len(rows)}')

In [ ]:
# ============================================================
# Build & save submission
# ============================================================
submission = pd.DataFrame(rows, columns=['row_id'] + species_cols)

submission = (
    submission
    .set_index('row_id')
    .reindex(sample_sub['row_id'], fill_value=0.0)
    .reset_index()
)

out_path = f'{OUTPUT_DIR}/submission.csv'
submission.to_csv(out_path, index=False)

# Verify
assert len(submission) == len(sample_sub), 'Row count mismatch'
assert list(submission.columns) == list(sample_sub.columns), 'Column mismatch'
assert submission.isnull().sum().sum() == 0, 'NaN found'

score_vals = submission.drop('row_id', axis=1).values
print(f'Submission saved: {out_path}')
print(f'  Rows: {len(submission)}')
print(f'  Mean: {score_vals.mean():.6f}')
print(f'  Max:  {score_vals.max():.4f}')
print(f'  Zero: {(score_vals == 0.0).mean():.2%}')
submission.head(3)